# Example: systematic 3D models

This notebook generates a **family of parametric 3D models** and exports them as JSON for the website (`web/`).

Each model is a twisted tower built from boxes. The parameters (`levels`, `twist`) are swept systematically,
so the site shows one model per parameter combination.

Replace the generator with your own logic: the website only depends on the JSON format described at the bottom.

In [ ]:
# Parameters (papermill can override these in CI: tag this cell "parameters")
output_path = "../data/models.json"
levels_options = [4, 8, 12]
twist_options = [0, 15, 30]   # degrees of rotation per level
boxes_per_level = 6

In [ ]:
import json
import math
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
def generate_tower(levels: int, twist_deg: float, boxes_per_level: int, radius: float = 1.6):
    """Return a list of box parts. Each part has a position, size, rotation and a 0..1 value used for colour."""
    parts = []
    level_height = 0.5
    for level in range(levels):
        base_angle = math.radians(twist_deg * level)
        # Taper the tower slightly towards the top
        r = radius * (1 - 0.35 * level / max(levels - 1, 1))
        for i in range(boxes_per_level):
            angle = base_angle + 2 * math.pi * i / boxes_per_level
            parts.append({
                "x": round(r * math.cos(angle), 4),
                "y": round(level * level_height + level_height / 2, 4),
                "z": round(r * math.sin(angle), 4),
                "w": 0.8, "h": level_height * 0.9, "d": 0.4,
                "ry": round(-angle, 4),
                "value": round(level / max(levels - 1, 1), 4),
                "level": level,
            })
    return parts


models = []
for levels in levels_options:
    for twist in twist_options:
        parts = generate_tower(levels, twist, boxes_per_level)
        models.append({
            "id": f"tower-L{levels}-T{twist}",
            "params": {"levels": levels, "twist": twist, "boxes_per_level": boxes_per_level},
            "parts": parts,
        })

pd.DataFrame([{**m["params"], "id": m["id"], "parts": len(m["parts"])} for m in models])

In [ ]:
out = Path(output_path)
out.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "source": "notebooks/example_models.ipynb",
    "models": models,
}
out.write_text(json.dumps(payload, indent=1))
print(f"Wrote {len(models)} models to {out.resolve()}")

## Output format (`data/models.json`)

```json
{
  "generated_at": "2026-01-01T00:00:00+00:00",
  "source": "notebooks/example_models.ipynb",
  "models": [
    {
      "id": "tower-L4-T0",
      "params": { "levels": 4, "twist": 0 },
      "parts": [ { "x": 0, "y": 0, "z": 0, "w": 1, "h": 1, "d": 1, "ry": 0, "value": 0.5 } ]
    }
  ]
}
```

- `x, y, z`: centre of the box (y is up), `w, h, d`: size, `ry`: rotation around the vertical axis in radians.
- `value`: 0..1, mapped to colour in the viewer.
- `params`: free-form; shown in the website's info panel.